Ejemplo de codigo para representar un Robot con objetos STL 
Traduccion del Script original Robot_2GDL_STL_Animacion.m de la toolbox "Simple_Robotics_Toolbox_oct2026"

In [1]:
"""Traduccion a Python del script de animacion del robot RR de MATLAB."""

from pathlib import Path

import asyncio
from IPython.display import display
import numpy as np
import SimpleRobotics as rt

# Parametros del robot
L1 = 0.200
L2 = 0.150
codo = 1

# Posicion inicial del robot (MATLAB: Xd=0.100, Yd=0.200)
theta1, theta2 = rt.IK_RR(0.100, 0.200, L1, L2, codo)
Q = np.rad2deg([theta1, theta2])
TK = rt.DK_RR(theta1, theta2, L1, L2)
Xr_inicial = TK[0, 3]
Yr_inicial = TK[1, 3]

# Crear la escena del robot una sola vez
# carpeta_stl = (Path(__file__).resolve().parents[1] / "STL_RR") este comando se utiliza en un archivo .py
carpeta_stl = Path.cwd().parent /"STL_RR"
scene, H02 = rt.Renderiza_Robot_RR(
    theta1,
    theta2,
    carpeta_stl,
    mostrar=False,
)

plotter = scene.plotter

# Vista aproximada a view([70, 25]) del script MATLAB
plotter.camera_position = [
    (0.28, 0.77, 0.38),
    (0.0, 0.0, 0.0),
    (0.0, 0.0, 1.0),
]

# Trayectoria cartesiana original de MATLAB
Yd = [0.1, 0.1, 0.0, -0.1, -0.1, 0.0, 0.1]
Xd = [0.1, 0.2, 0.3, 0.2, 0.1, 0.2, 0.1]

# Cinematica inversa de los puntos de la trayectoria
Qd = [rt.IK_RR(x, y, L1, L2, codo) for x, y in zip(Xd, Yd)]

# Trayectoria del efector final (MATLAB: Xr=[], Yr=[], Zr=[])
Xr = []
Yr = []
Zr = []

# Animacion continua: 30 cuadros/s y 0.3 s por segmento
FPS = 120
MUESTRAS = round(FPS * 0.3)

def animar(step):
    """Actualiza la pose del robot en cada cuadro de la animacion."""
    vc = step // MUESTRAS
    k = step % MUESTRAS + 1

    # Configuracion inicial y final del segmento actual
    if vc == 0:
        q0 = np.array([theta1, theta2])
    else:
        q0 = np.array(Qd[vc - 1])

    qf = np.array(Qd[vc])

    # Interpolacion articular para mostrar el movimiento continuo
    q = q0 + (qf - q0) * k / MUESTRAS

    # Actualizar el robot y obtener la matriz H02
    H02 = rt.actualizar_pose_robot_rr(scene, q[0], q[1])

    # Al terminar cada segmento, registrar la posicion del efector final
    if k == MUESTRAS:
        Xr.append(H02[0, 3])
        Yr.append(H02[1, 3])
        Zr.append(0.0)

        puntos = np.column_stack((Xr, Yr, Zr))

        if len(puntos) > 1:
            plotter.add_lines(
                puntos,
                color="red",
                width=2,
                connected=True,
                name="trayectoria",
            )

        plotter.add_points(
            puntos,
            color="blue",
            point_size=12,
            point_shape="asterisk",
            name="puntos_efector",
        )

    scene.render()
# Mostrar la escena interactiva en Jupyter
viewer = scene.show(
    auto_close=False,
    jupyter_backend="trame",
    return_viewer=True,
)

display(viewer)


# Reproducir la animación
async def reproducir():

    await asyncio.sleep(0.5)

    for step in range(len(Xd) * MUESTRAS):

        animar(step)

        # Actualizar la visualización de Trame
        viewer.viewer.update()

        await asyncio.sleep(1 / FPS)


await reproducir()